# 📈 Comparação Geral — LOOCV
**Todos os 6 modelos com Leave-One-Out Cross-Validation**

## 1. Bibliotecas

In [ ]:
!pip install catboost lightgbm openpyxl --quiet
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
SEED = 42; np.random.seed(SEED)

def calc_metrics(y_true, y_pred, model_name='Modelo'):
    r2    = r2_score(y_true, y_pred)
    rmse  = np.sqrt(mean_squared_error(y_true, y_pred))
    mae   = mean_absolute_error(y_true, y_pred)
    nrmse = rmse / (y_true.max() - y_true.min()) * 100
    bias  = np.mean(y_pred - y_true)
    mape  = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    rpd   = y_true.std() / rmse
    mu_t, mu_p = y_true.mean(), y_pred.mean()
    s2_t = np.var(y_true); s2_p = np.var(y_pred)
    s_tp = np.mean((y_true - mu_t) * (y_pred - mu_p))
    ccc  = (2 * s_tp) / (s2_t + s2_p + (mu_t - mu_p) ** 2)
    print(f'\n=== {model_name} — LOOCV ===')
    print(f'  R²={r2:.4f}  CCC={ccc:.4f}  RMSE={rmse:.4f}  MAE={mae:.4f}')
    print(f'  NRMSE={nrmse:.2f}%  MAPE={mape:.2f}%  Bias={bias:.4f}  RPD={rpd:.4f}')
    return {'Modelo': model_name,
            'R2': round(r2,4), 'CCC': round(ccc,4), 'RMSE': round(rmse,4),
            'MAE': round(mae,4), 'NRMSE(%)': round(nrmse,2),
            'MAPE(%)': round(mape,2), 'Bias': round(bias,4), 'RPD': round(rpd,4)}

print('✅ Bibliotecas carregadas!')

## 2. Carregar Dados

In [ ]:
import os, io
IN_COLAB = False
try:
    import google.colab; IN_COLAB = True
except ImportError:
    pass

if IN_COLAB:
    from google.colab import files
    print("📂 Selecione o arquivo Excel ou CSV:")
    uploaded = files.upload()
    fname = list(uploaded.keys())[0]
    ext = fname.split('.')[-1].lower()
    df = pd.read_excel(io.BytesIO(uploaded[fname])) if ext in ['xlsx','xls'] else pd.read_csv(io.BytesIO(uploaded[fname]))
else:
    FILE_PATH = 'só_indices.xlsx'  # ⚠️ ajuste se necessário
    assert os.path.exists(FILE_PATH), f"Arquivo não encontrado: {FILE_PATH}"
    df = pd.read_excel(FILE_PATH)

TARGET    = 'Biomass_g'
DROP_COLS = ['Flight_ID']
FEATURES  = [c for c in df.columns if c not in [TARGET] + DROP_COLS]
df = df[FEATURES + [TARGET]].dropna().reset_index(drop=True)
X  = df[FEATURES].values
y  = df[TARGET].values
print(f'✅ Amostras: {len(y)}  |  Features: {len(FEATURES)}')
df.head()

## 3. Rodar Todos os Modelos com LOOCV

In [ ]:
from catboost import CatBoostRegressor
import lightgbm as lgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.kernel_ridge import KernelRidge
from sklearn.cross_decomposition import PLSRegression
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel

models = {
    'CatBoost':     (Pipeline([('s', RobustScaler()), ('model', CatBoostRegressor(iterations=300, learning_rate=0.05, depth=4, l2_leaf_reg=3, random_seed=SEED, verbose=0))]),     '#FF8C00'),
    'RandomForest': (Pipeline([('s', RobustScaler()), ('model', RandomForestRegressor(n_estimators=300, random_state=SEED, n_jobs=-1))]),                                           '#2E8B57'),
    'LightGBM':     (Pipeline([('s', RobustScaler()), ('model', lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, num_leaves=15, min_child_samples=5, reg_lambda=1.0, random_state=SEED, verbose=-1))]), '#DAA520'),
    'KRR':          (Pipeline([('s', RobustScaler()), ('model', KernelRidge(kernel='rbf', alpha=0.1, gamma=0.1))]),                                                                 '#4169E1'),
    'GPR':          (Pipeline([('s', RobustScaler()), ('model', GaussianProcessRegressor(kernel=1.0*Matern(nu=2.5)+WhiteKernel(1e-2), alpha=1e-2, normalize_y=True, random_state=SEED))]), '#9370DB'),
    'PLS':          (Pipeline([('s', RobustScaler()), ('model', PLSRegression(n_components=5, max_iter=1000))]),                                                                    '#DC143C'),
}

loo = LeaveOneOut()
all_results = []

for name, (pipe, color) in models.items():
    print(f'▶ {name}...')
    y_pred = cross_val_predict(pipe, X, y, cv=loo)
    if hasattr(y_pred, 'ravel'): y_pred = y_pred.ravel()
    m = calc_metrics(y, y_pred, name)
    m['color'] = color
    all_results.append(m)

comp_df = pd.DataFrame(all_results)
cols = ['Modelo','R2','CCC','RMSE','MAE','NRMSE(%)','MAPE(%)','Bias','RPD']
print('\n📊 Comparação Geral — LOOCV:')
print(comp_df[cols].to_string(index=False))
comp_df[cols].to_csv('Comparacao_LOOCV_metricas.csv', index=False)

## 4. Gráfico Comparativo

In [ ]:
# ── Gráfico comparativo de barras ────────────────────────────────────────────
plt.rcParams['font.family'] = 'Times New Roman'
metrics_cols = ['R2','CCC','RMSE','MAE','NRMSE(%)','MAPE(%)','Bias','RPD']
bar_colors   = comp_df['color'].tolist()
model_names  = comp_df['Modelo'].tolist()

fig, axes = plt.subplots(2, 4, figsize=(22, 10))
for ax, col in zip(axes.ravel(), metrics_cols):
    bars = ax.bar(model_names, comp_df[col].values, color=bar_colors, edgecolor='white')
    ax.set_title(col, fontsize=13, fontweight='bold')
    ax.tick_params(axis='x', rotation=30, labelsize=11)
    ax.tick_params(axis='y', labelsize=11)
    ax.bar_label(bars, fmt='%.3f', fontsize=9, padding=3)
    ax.grid(axis='y', alpha=0.3)
    for spine in ['top','right']: ax.spines[spine].set_visible(False)

plt.suptitle('Comparação de Modelos — LOOCV', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('Comparacao_LOOCV_barplot.png', dpi=180, bbox_inches='tight')
plt.show()

## 5. Exportar

In [ ]:
import zipfile
files_to_zip = [f for f in os.listdir('.') if f.startswith('Comparacao') and (f.endswith('.csv') or f.endswith('.png'))]
zip_name = 'Comparacao_LOOCV_resultados.zip'
with zipfile.ZipFile(zip_name, 'w') as zf:
    for f in files_to_zip: zf.write(f); print(f'  + {f}')
if IN_COLAB:
    from google.colab import files as colab_files
    colab_files.download(zip_name)
else:
    print('✅ Salvo na pasta do notebook.')